In [ ]:
!nvidia-smi

Thu Oct 24 18:43:07 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   46C    P8               9W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [7]:
# Pip install method (recommended)

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 39.0/112.6 GB disk)


In [6]:
from ultralytics import YOLO

from IPython.display import display, Image

ModuleNotFoundError: No module named 'ultralytics'

In [20]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="КЛЮЧ")
project = rf.workspace("РАБОЧЕЕ ПРОСТРАНСТВО").project("ПРОЕКТ")
version = project.version(3)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...


In [1]:
!pip install roboflow

In [26]:
# Удаляем всё разом
import shutil
shutil.rmtree('/content/datasets', ignore_errors=True)
shutil.rmtree('/content/Project_Test_1-1', ignore_errors=True)
shutil.rmtree('/content/Project_Test_1-3', ignore_errors=True)

print("✅ Всё удалено!")

✅ Всё удалено!


In [2]:
import os
import shutil
from roboflow import Roboflow
import yaml

print("=" * 70)
print("🗑️  ОЧИСТКА И ПЕРЕЗАГРУЗКА ДАТАСЕТА")
print("=" * 70)

# ========================================
# 1. УДАЛЕНИЕ СТАРЫХ ДАТАСЕТОВ
# ========================================
print("\n🔍 Поиск старых датасетов...")

folders_to_clean = [
    "/content/datasets",
    "/content/Project_Test_1-1",
    "/content/Project_Test_1-3",
]

total_deleted = 0
for folder in folders_to_clean:
    if os.path.exists(folder):
        # Показываем размер
        size_mb = sum(
            os.path.getsize(os.path.join(dirpath, filename))
            for dirpath, dirnames, filenames in os.walk(folder)
            for filename in filenames
        ) / (1024 * 1024)

        print(f"\n📁 {folder}")
        print(f"   Размер: {size_mb:.2f} MB")

        # Удаляем
        try:
            shutil.rmtree(folder)
            print(f"   ✅ Удалено!")
            total_deleted += size_mb
        except Exception as e:
            print(f"   ❌ Ошибка: {e}")
    else:
        print(f"\n⏭️  {folder} - уже не существует")

print(f"\n💾 Освобождено: {total_deleted:.2f} MB")

# ========================================
# 2. ПРАВИЛЬНАЯ ЗАГРУЗКА ИЗ ROBOFLOW
# ========================================
print("\n" + "=" * 70)
print("📥 ЗАГРУЗКА ДАТАСЕТА (ПРАВИЛЬНО)")
print("=" * 70)

# Параметры (ТВОИ ДАННЫЕ)
API_KEY = ""  # <-- ВСТАВЬ СЮДА!
WORKSPACE = ""
PROJECT = ""
VERSION = 3

# ПРАВИЛЬНЫЙ путь - БЕЗ лишних вложений
CORRECT_PATH = "/content/yolo_dataset"

print(f"\n📋 Параметры:")
print(f"   Workspace: {WORKSPACE}")
print(f"   Project: {PROJECT}")
print(f"   Version: {VERSION}")
print(f"   Папка: {CORRECT_PATH}")

# Создаем папку
os.makedirs(CORRECT_PATH, exist_ok=True)

# Авторизация
print("\n🔐 Подключение к Roboflow...")
rf = Roboflow(api_key=API_KEY)

# Загрузка
print("📦 Загрузка проекта...")
project = rf.workspace(WORKSPACE).project(PROJECT)

print("⬇️  Скачивание датасета...")
dataset = project.version(VERSION).download(
    "yolov8",
    location=CORRECT_PATH,
    overwrite=True  # Перезаписываем если есть
)

# ========================================
# 3. ПРОВЕРКА РЕЗУЛЬТАТА
# ========================================
print("\n" + "=" * 70)
print("✅ ПРОВЕРКА ЗАГРУЖЕННОГО ДАТАСЕТА")
print("=" * 70)

# Находим data.yaml
data_yaml_path = None
for root, dirs, files in os.walk(CORRECT_PATH):
    for file in files:
        if file == "data.yaml":
            data_yaml_path = os.path.join(root, file)
            break
    if data_yaml_path:
        break

if not data_yaml_path:
    print("❌ data.yaml не найден!")
    print("\n📁 Содержимое папки:")
    for item in os.listdir(CORRECT_PATH):
        print(f"   • {item}")
else:
    print(f"✅ data.yaml найден: {data_yaml_path}")

    # Читаем конфигурацию
    with open(data_yaml_path, 'r') as f:
        config = yaml.safe_load(f)

    print(f"\n📄 Конфигурация:")
    print(f"   Классов: {config.get('nc', 'N/A')}")
    print(f"   Имена: {config.get('names', [])}")

    # Определяем базовую папку датасета
    base_dir = os.path.dirname(data_yaml_path)

    # Проверяем splits
    print(f"\n📊 Статистика по splits:")
    splits = ['train', 'valid', 'test']

    for split in splits:
        img_dir = os.path.join(base_dir, split, 'images')
        lbl_dir = os.path.join(base_dir, split, 'labels')

        if os.path.exists(img_dir):
            img_count = len([f for f in os.listdir(img_dir)
                           if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
            lbl_count = len([f for f in os.listdir(lbl_dir)
                           if f.lower().endswith('.txt')]) if os.path.exists(lbl_dir) else 0

            status = "✅" if img_count > 0 else "⚠️"
            print(f"   {status} {split:6} → {img_count:4} изображений, {lbl_count:4} аннотаций")
        else:
            print(f"   ❌ {split:6} → папка не существует")

    # ========================================
    # 4. ВЫВОД ДЛЯ ОБУЧЕНИЯ
    # ========================================
    print("\n" + "=" * 70)
    print("🚀 ГОТОВО К ОБУЧЕНИЮ!")
    print("=" * 70)

    print(f"\n📝 Используй этот код:\n")
    print(f"""from ultralytics import YOLO

# Путь к датасету
dataset_path = '{data_yaml_path}'

# Загрузка модели
model = YOLO('yolov8l.pt')

# Обучение
results = model.train(
    data=dataset_path,
    epochs=100,
    imgsz=1280,
    batch=6,
    device=0,
    val=True,
    patience=20,
    save=True,
    save_period=10,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.001,
    amp=True,
    plots=True
)

print(f"📁 Результаты: {{results.save_dir}}")
print(f"🏆 Лучшая модель: {{results.save_dir}}/weights/best.pt")
""")

print("=" * 70)

🗑️  ОЧИСТКА И ПЕРЕЗАГРУЗКА ДАТАСЕТА

🔍 Поиск старых датасетов...

⏭️  /content/datasets - уже не существует

⏭️  /content/Project_Test_1-1 - уже не существует

⏭️  /content/Project_Test_1-3 - уже не существует

💾 Освобождено: 0.00 MB

📥 ЗАГРУЗКА ДАТАСЕТА (ПРАВИЛЬНО)

📋 Параметры:
   Workspace: tets1-k0emt
   Project: project_test_1-qlnqb
   Version: 3
   Папка: /content/yolo_dataset

🔐 Подключение к Roboflow...
📦 Загрузка проекта...
loading Roboflow workspace...
loading Roboflow project...
⬇️  Скачивание датасета...



Extracting Dataset Version Zip to /content/yolo_dataset in yolov8:: 100%|██████████| 348/348 [00:00<00:00, 3899.47it/s]



✅ ПРОВЕРКА ЗАГРУЖЕННОГО ДАТАСЕТА
✅ data.yaml найден: /content/yolo_dataset/data.yaml

📄 Конфигурация:
   Классов: 8
   Имена: ['detail', 'legend', 'note', 'plan', 'section', 'site_plan', 'stamp', 'table']

📊 Статистика по splits:
   ✅ train  →  171 изображений,  171 аннотаций
   ❌ valid  → папка не существует
   ❌ test   → папка не существует

🚀 ГОТОВО К ОБУЧЕНИЮ!

📝 Используй этот код:

from ultralytics import YOLO

# Путь к датасету
dataset_path = '/content/yolo_dataset/data.yaml'

# Загрузка модели
model = YOLO('yolov8l.pt')

# Обучение
results = model.train(
    data=dataset_path,
    epochs=100,
    imgsz=1280,
    batch=6,
    device=0,
    val=True,
    patience=20,
    save=True,
    save_period=10,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.001,
    amp=True,
    plots=True
)

print(f"📁 Результаты: {results.save_dir}")
print(f"🏆 Лучшая модель: {results.save_dir}/weights/best.pt")



In [3]:
import os
import shutil
import yaml
from pathlib import Path
from sklearn.model_selection import train_test_split

print("=" * 70)
print("📊 СОЗДАНИЕ TRAIN / VALID / TEST SPLITS")
print("=" * 70)

# Путь к датасету
dataset_path = '/content/yolo_dataset'
data_yaml = os.path.join(dataset_path, 'data.yaml')

# Проверяем текущую структуру
print("\n🔍 Текущая структура:")
train_img_dir = os.path.join(dataset_path, 'train', 'images')
train_lbl_dir = os.path.join(dataset_path, 'train', 'labels')

if not os.path.exists(train_img_dir):
    print("❌ Папка train/images не существует!")
    # Ищем где реально лежат файлы
    print("\n🔎 Поиск изображений...")
    for root, dirs, files in os.walk(dataset_path):
        images = [f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        if images:
            print(f"   Найдено {len(images)} изображений в: {root}")
            train_img_dir = root
            # Предполагаем что labels рядом
            train_lbl_dir = root.replace('images', 'labels')
            if not os.path.exists(train_lbl_dir):
                train_lbl_dir = os.path.join(os.path.dirname(root), 'labels')
            break

# Получаем список всех изображений
all_images = [f for f in os.listdir(train_img_dir)
              if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

print(f"\n📸 Найдено изображений: {len(all_images)}")

if len(all_images) == 0:
    print("❌ ОШИБКА: Нет изображений для разделения!")
    exit()

# ========================================
# РАЗДЕЛЕНИЕ: 70% train, 20% valid, 10% test
# ========================================
print("\n🔀 Разделение датасета...")
print("   📊 Train: 70%")
print("   📊 Valid: 20%")
print("   📊 Test:  10%")

# Сначала отделяем test (10%)
train_valid_imgs, test_imgs = train_test_split(
    all_images, test_size=0.10, random_state=42
)

# Потом из оставшихся делаем train/valid (70/20 от исходного)
train_imgs, valid_imgs = train_test_split(
    train_valid_imgs, test_size=0.222, random_state=42  # 0.222 * 0.9 ≈ 0.2
)

print(f"\n✅ Разделение выполнено:")
print(f"   Train: {len(train_imgs)} изображений ({len(train_imgs)/len(all_images)*100:.1f}%)")
print(f"   Valid: {len(valid_imgs)} изображений ({len(valid_imgs)/len(all_images)*100:.1f}%)")
print(f"   Test:  {len(test_imgs)} изображений ({len(test_imgs)/len(all_images)*100:.1f}%)")

# ========================================
# СОЗДАНИЕ ПАПОК
# ========================================
print("\n📁 Создание структуры папок...")

splits = {
    'train': train_imgs,
    'valid': valid_imgs,
    'test': test_imgs
}

for split_name, img_list in splits.items():
    # Создаем папки
    split_img_dir = os.path.join(dataset_path, split_name, 'images')
    split_lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    os.makedirs(split_img_dir, exist_ok=True)
    os.makedirs(split_lbl_dir, exist_ok=True)

    print(f"   ✅ {split_name}/images")
    print(f"   ✅ {split_name}/labels")

# ========================================
# ПЕРЕМЕЩЕНИЕ ФАЙЛОВ
# ========================================
print("\n🚚 Перемещение файлов...")

# Сначала очищаем train (если там были все файлы)
temp_img_dir = os.path.join(dataset_path, '_temp_images')
temp_lbl_dir = os.path.join(dataset_path, '_temp_labels')

# Копируем все в temp
if os.path.exists(train_img_dir):
    shutil.copytree(train_img_dir, temp_img_dir, dirs_exist_ok=True)
if os.path.exists(train_lbl_dir):
    shutil.copytree(train_lbl_dir, temp_lbl_dir, dirs_exist_ok=True)

# Очищаем старые папки train
for folder in [train_img_dir, train_lbl_dir]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder, exist_ok=True)

# Распределяем файлы
for split_name, img_list in splits.items():
    split_img_dir = os.path.join(dataset_path, split_name, 'images')
    split_lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    for img_name in img_list:
        # Определяем имя label файла
        base_name = os.path.splitext(img_name)[0]
        lbl_name = base_name + '.txt'

        # Источники (из temp)
        src_img = os.path.join(temp_img_dir, img_name)
        src_lbl = os.path.join(temp_lbl_dir, lbl_name)

        # Назначения
        dst_img = os.path.join(split_img_dir, img_name)
        dst_lbl = os.path.join(split_lbl_dir, lbl_name)

        # Копируем
        if os.path.exists(src_img):
            shutil.copy2(src_img, dst_img)

        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)

    print(f"   ✅ {split_name}: {len(img_list)} файлов перемещено")

# Удаляем temp папки
shutil.rmtree(temp_img_dir, ignore_errors=True)
shutil.rmtree(temp_lbl_dir, ignore_errors=True)

# ========================================
# ОБНОВЛЕНИЕ data.yaml
# ========================================
print("\n📝 Обновление data.yaml...")

with open(data_yaml, 'r') as f:
    config = yaml.safe_load(f)

# Обновляем пути
config['train'] = 'train/images'
config['val'] = 'valid/images'
config['test'] = 'test/images'

# Сохраняем
with open(data_yaml, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("   ✅ data.yaml обновлен")

# ========================================
# ФИНАЛЬНАЯ ПРОВЕРКА
# ========================================
print("\n" + "=" * 70)
print("✅ ИТОГОВАЯ СТРУКТУРА")
print("=" * 70)

print(f"\n📄 Конфигурация data.yaml:")
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

print("\n📊 Статистика по splits:")
for split_name in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, split_name, 'images')
    lbl_dir = os.path.join(dataset_path, split_name, 'labels')

    img_count = len([f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) if os.path.exists(img_dir) else 0
    lbl_count = len([f for f in os.listdir(lbl_dir)
                    if f.lower().endswith('.txt')]) if os.path.exists(lbl_dir) else 0

    status = "✅" if img_count > 0 else "❌"
    print(f"{status} {split_name.upper():6} → Images: {img_count:4} | Labels: {lbl_count:4}")

# ========================================
# КОД ДЛЯ ОБУЧЕНИЯ
# ========================================
print("\n" + "=" * 70)
print("🚀 ГОТОВО! ЗАПУСКАЙ ОБУЧЕНИЕ")
print("=" * 70)

print(f"""
from ultralytics import YOLO

# Путь к датасету
dataset_path = '{data_yaml}'

# Загрузка модели YOLOv8 Large
model = YOLO('yolov8l.pt')

# Обучение
results = model.train(
    data=dataset_path,
    epochs=100,
    imgsz=1280,
    batch=6,
    device=0,

    # Валидация и сохранение
    val=True,
    patience=20,
    save=True,
    save_period=10,

    # Оптимизатор
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.001,
    weight_decay=0.0005,
    warmup_epochs=5,

    # Аугментации (минимальные для чертежей)
    hsv_h=0.005,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=0,
    translate=0.05,
    scale=0.3,
    shear=0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,

    # Дополнительно
    close_mosaic=20,
    amp=True,
    plots=True,
    verbose=True
)

print("\\n" + "="*50)
print("🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*50)
print(f"📁 Результаты: {{results.save_dir}}")
print(f"🏆 Лучшая модель: {{results.save_dir}}/weights/best.pt")
""")

print("=" * 70)

📊 СОЗДАНИЕ TRAIN / VALID / TEST SPLITS

🔍 Текущая структура:

📸 Найдено изображений: 171

🔀 Разделение датасета...
   📊 Train: 70%
   📊 Valid: 20%
   📊 Test:  10%

✅ Разделение выполнено:
   Train: 119 изображений (69.6%)
   Valid: 34 изображений (19.9%)
   Test:  18 изображений (10.5%)

📁 Создание структуры папок...
   ✅ train/images
   ✅ train/labels
   ✅ valid/images
   ✅ valid/labels
   ✅ test/images
   ✅ test/labels

🚚 Перемещение файлов...
   ✅ train: 119 файлов перемещено
   ✅ valid: 34 файлов перемещено
   ✅ test: 18 файлов перемещено

📝 Обновление data.yaml...
   ✅ data.yaml обновлен

✅ ИТОГОВАЯ СТРУКТУРА

📄 Конфигурация data.yaml:
names:
- detail
- legend
- note
- plan
- section
- site_plan
- stamp
- table
nc: 8
roboflow:
  license: MIT
  project: project_test_1-qlnqb
  url: https://universe.roboflow.com/tets1-k0emt/project_test_1-qlnqb/dataset/3
  version: 3
  workspace: tets1-k0emt
test: test/images
train: train/images
val: valid/images


📊 Статистика по splits:
✅ TRAIN  → 

In [4]:
# Добавьте это ДО начала обучения для диагностики
import yaml

with open('/content/yolo_dataset/data.yaml', 'r') as f:
    config = yaml.safe_load(f)
    print("📋 Конфигурация data.yaml:")
    print(f"   Classes: {config['nc']}")
    print(f"   Names: {config['names']}")
    print(f"   Train: {config['train']}")
    print(f"   Valid: {config['val']}")

📋 Конфигурация data.yaml:
   Classes: 8
   Names: ['detail', 'legend', 'note', 'plan', 'section', 'site_plan', 'stamp', 'table']
   Train: train/images
   Valid: valid/images


In [5]:
import yaml
import os

# Путь к data.yaml
dataset_path = '/content/yolo_dataset/data.yaml'

# Читаем конфиг
with open(dataset_path, 'r') as f:
    config = yaml.safe_load(f)

# Исправляем пути на абсолютные
base_path = '/content/yolo_dataset'
config['train'] = f'{base_path}/train/images'
config['val'] = f'{base_path}/valid/images'
config['test'] = f'{base_path}/test/images'

# Сохраняем обновленный конфиг
with open(dataset_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml обновлен с абсолютными путями:")
print(f"   Train: {config['train']}")
print(f"   Valid: {config['val']}")
print(f"   Test: {config['test']}")
print()

# Проверка существования папок
for split in ['train', 'valid', 'test']:
    img_path = f'{base_path}/{split}/images'
    lbl_path = f'{base_path}/{split}/labels'

    img_count = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))])
    lbl_count = len([f for f in os.listdir(lbl_path) if f.endswith('.txt')])

    print(f"📊 {split.upper()}: {img_count} images, {lbl_count} labels")

✅ data.yaml обновлен с абсолютными путями:
   Train: /content/yolo_dataset/train/images
   Valid: /content/yolo_dataset/valid/images
   Test: /content/yolo_dataset/test/images

📊 TRAIN: 119 images, 119 labels
📊 VALID: 34 images, 34 labels
📊 TEST: 18 images, 18 labels


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# В параметрах model.train():
project='/content/drive/MyDrive/yolo_training',  # Сохранение на Drive
save_period=5,  # Сохранять каждые 5 эпох

In [7]:
from ultralytics import YOLO
import yaml
import os

# ====== 1. ИСПРАВЛЕНИЕ ПУТЕЙ В data.yaml ======
dataset_path = '/content/yolo_dataset/data.yaml'
base_path = '/content/yolo_dataset'

with open(dataset_path, 'r') as f:
    config = yaml.safe_load(f)

config['train'] = f'{base_path}/train/images'
config['val'] = f'{base_path}/valid/images'
config['test'] = f'{base_path}/test/images'

with open(dataset_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Пути обновлены в data.yaml\n")

# ====== 2. ПРОВЕРКА ДАТАСЕТА ======
print("📊 Проверка датасета:")
for split in ['train', 'valid', 'test']:
    img_path = f'{base_path}/{split}/images'
    lbl_path = f'{base_path}/{split}/labels'

    if os.path.exists(img_path):
        img_count = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))])
        lbl_count = len([f for f in os.listdir(lbl_path) if f.endswith('.txt')])
        print(f"   {split.upper()}: {img_count} images, {lbl_count} labels")
    else:
        print(f"   ❌ {split.upper()}: папка не найдена!")

print("\n" + "="*70)
print("🚀 ЗАПУСК ОБУЧЕНИЯ")
print("="*70 + "\n")

# ====== 3. ОБУЧЕНИЕ ======
model = YOLO('yolov8l.pt')

results = model.train(
    data=dataset_path,

    # Основные параметры
    epochs=100,
    imgsz=1280,
    batch=6,
    device=0,

    # Валидация и сохранение
    val=True,
    patience=20,
    save=True,
    save_period=10,

    # Оптимизатор
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.001,
    weight_decay=0.0005,
    warmup_epochs=5,

    # Аугментации (минимальные для чертежей)
    hsv_h=0.005,
    hsv_s=0.3,
    hsv_v=0.2,
    degrees=0,
    translate=0.05,
    scale=0.3,
    shear=0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,

    # Дополнительно
    close_mosaic=20,
    amp=True,
    plots=True,
    verbose=True
)

print("\n" + "="*70)
print("🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*70)
print(f"📁 Результаты: {results.save_dir}")
print(f"🏆 Лучшая модель: {results.save_dir}/weights/best.pt")

✅ Пути обновлены в data.yaml

📊 Проверка датасета:
   TRAIN: 119 images, 119 labels
   VALID: 34 images, 34 labels
   TEST: 18 images, 18 labels

🚀 ЗАПУСК ОБУЧЕНИЯ



100%|██████████| 83.7M/83.7M [00:01<00:00, 68.9MB/s]


New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8l.pt, data=/content/yolo_dataset/data.yaml, epochs=100, time=None, patience=20, batch=6, imgsz=1280, save=True, save_period=10, cache=False, device=0, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=20, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save

100%|██████████| 755k/755k [00:00<00:00, 31.8MB/s]


Overriding model.yaml nc=80 with nc=8

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  3    279808  ultralytics.nn.modules.block.C2f             [128, 128, 3, True]           
  3                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  4                  -1  6   2101248  ultralytics.nn.modules.block.C2f             [256, 256, 6, True]           
  5                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  6                  -1  6   8396800  ultralytics.nn.modules.block.C2f             [512, 512, 6, True]           
  7                  -1  1   2360320  ultralytics

...

     68/100      14.9G     0.4553     0.3695     0.9557         45       1280: 100%|██████████| 20/20 [00:23<00:00,  1.15s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.16it/s]

                   all         34        220      0.951      0.928      0.956      0.804
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 48, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



68 epochs completed in 0.582 hours.
Optimizer stripped from runs/detect/train/weights/last.pt, 87.8MB
Optimizer stripped from runs/detect/train/weights/best.pt, 87.8MB

Validating runs/detect/train/weights/best.pt...
Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 268 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.04it/s]


                   all         34        220      0.964      0.938      0.991      0.831
                detail         14         64      0.984      0.935      0.988      0.846
                legend          2          2          1      0.666      0.995      0.821
                  note         27         36      0.946      0.969      0.981      0.839
                  plan         20         30          1      0.944      0.994       0.77
               section          1          2      0.889          1      0.995      0.597
             site_plan          8          8      0.931          1      0.995      0.901
                 stamp         33         33      0.964          1      0.988      0.979
                 table         18         45          1       0.99      0.995      0.894
Speed: 1.0ms preprocess, 66.4ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to runs/detect/train

🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!
📁 Результаты: runs/detect/train
🏆 Лучшая модель: runs/dete

In [11]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 19.0 MB/s eta 0:00:00


In [17]:
from ultralytics import YOLO
from google.colab import files

# Загружаем обученную модель
model = YOLO(f'{results.save_dir}/weights/best.pt')

# Экспортируем в ONNX с нужными параметрами
onnx_path = model.export(
    format='onnx',
    simplify=True      # ← упрощение графа (очень рекомендуется)
)

print(f"✅ Модель экспортирована в ONNX: {onnx_path}")

# Скачивание PyTorch модели (.pt)
print("\n📥 Скачивание PyTorch модели...")
files.download(f'{results.save_dir}/weights/best.pt')

# Скачивание ONNX модели
print("\n📥 Скачивание ONNX модели...")
files.download(onnx_path)

print("\n✨ Обе модели скачаны!")
print(f"📦 PyTorch: best.pt")
print(f"📦 ONNX: {onnx_path}")

Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon 2.00GHz)
Model summary (fused): 268 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 12, 33600) (83.7 MB)

ONNX: starting export with onnx 1.20.1 opset 10...


W0122 12:41:57.832000 3251 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter

Applied 1 of general pattern rewrite rules.
ONNX: slimming with onnxslim 0.1.34...
ONNX: export success ✅ 17.6s, saved as 'runs/detect/train/weights/best.onnx' (166.8 MB)

Export complete (33.3s)
Results saved to /content/runs/detect/train/weights
Predict:         yolo predict task=detect model=runs/detect/train/weights/best.onnx imgsz=1280  
Validate:        yolo val task=detect model=runs/detect/train/weights/best.onnx imgsz=1280 data=/content/yolo_dataset/data.yaml  
Visualize:       https://netron.app
✅ Модель экспортирована в ONNX: runs/detect/train/weights/best.onnx

📥 Скачивание PyTorch модели...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Скачивание ONNX модели...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ Обе модели скачаны!
📦 PyTorch: best.pt
📦 ONNX: runs/detect/train/weights/best.onnx


In [18]:
from ultralytics import YOLO
from google.colab import files

# Загружаем обученную модель
model = YOLO(f'{results.save_dir}/weights/best.pt')

# Экспортируем в ONNX с нужными параметрами
onnx_path = model.export(
    format='onnx',
    simplify=True,     # упрощение графа — обязательно рекомендуется
    half=True          # ← добавляем FP16 (half-precision)
)

print(f"✅ Модель экспортирована в ONNX: {onnx_path}")

# Скачивание PyTorch модели (.pt)
print("\n📥 Скачивание PyTorch модели...")
files.download(f'{results.save_dir}/weights/best.pt')

# Скачивание ONNX модели
print("\n📥 Скачивание ONNX модели...")
files.download(onnx_path)

print("\n✨ Обе модели скачаны!")
print(f"📦 PyTorch: best.pt")
print(f"📦 ONNX: {onnx_path}")

Ultralytics YOLOv8.2.103 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon 2.00GHz)
WARNING ⚠️ half=True only compatible with GPU export, i.e. use device=0
Model summary (fused): 268 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 12, 33600) (83.7 MB)

ONNX: starting export with onnx 1.20.1 opset 10...


W0122 12:45:27.525000 3251 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 10 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter

Applied 1 of general pattern rewrite rules.
ONNX: slimming with onnxslim 0.1.34...
ONNX: export success ✅ 25.7s, saved as 'runs/detect/train/weights/best.onnx' (166.8 MB)

Export complete (41.3s)
Results saved to /content/runs/detect/train/weights
Predict:         yolo predict task=detect model=runs/detect/train/weights/best.onnx imgsz=1280  
Validate:        yolo val task=detect model=runs/detect/train/weights/best.onnx imgsz=1280 data=/content/yolo_dataset/data.yaml  
Visualize:       https://netron.app
✅ Модель экспортирована в ONNX: runs/detect/train/weights/best.onnx

📥 Скачивание PyTorch модели...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Скачивание ONNX модели...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✨ Обе модели скачаны!
📦 PyTorch: best.pt
📦 ONNX: runs/detect/train/weights/best.onnx


In [19]:
import os
import yaml
from pathlib import Path
import shutil

# ===== 1. ПРОВЕРКА СТРУКТУРЫ ДАТАСЕТА =====
dataset_path = '/content/datasets/datasets/Project_Test_1-1/data.yaml'
base_dir = os.path.dirname(dataset_path)

print("=" * 60)
print("🔍 ДИАГНОСТИКА ДАТАСЕТА")
print("=" * 60)

# Читаем data.yaml
with open(dataset_path, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\n📄 Содержимое data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

# Проверяем пути
required_dirs = {
    'train/images': os.path.join(base_dir, 'train', 'images'),
    'train/labels': os.path.join(base_dir, 'train', 'labels'),
    'valid/images': os.path.join(base_dir, 'valid', 'images'),
    'valid/labels': os.path.join(base_dir, 'valid', 'labels'),
}

print(f"\n📂 Проверка структуры папок:")
print("-" * 60)

missing_dirs = []
for name, path in required_dirs.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {name:20} -> {path}")
    if exists:
        count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f"   └─ Файлов: {count}")
    else:
        missing_dirs.append(name)

# ===== 2. ПОИСК РЕАЛЬНЫХ ФАЙЛОВ =====
print(f"\n🔎 Поиск всех изображений и аннотаций в {base_dir}:")
print("-" * 60)

all_images = []
all_labels = []

for root, dirs, files in os.walk(base_dir):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, base_dir)

        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            all_images.append(rel_path)
        elif file.lower().endswith('.txt') and file != 'data.yaml':
            all_labels.append(rel_path)

print(f"📸 Найдено изображений: {len(all_images)}")
print(f"🏷️  Найдено аннотаций: {len(all_labels)}")

if all_images:
    print("\nПримеры найденных изображений:")
    for img in all_images[:5]:
        print(f"  • {img}")

# ===== 3. АВТОМАТИЧЕСКОЕ ИСПРАВЛЕНИЕ =====
if missing_dirs:
    print(f"\n⚠️  Обнаружены отсутствующие папки!")
    print("🔧 Попытка автоматического исправления...\n")

    # Создаем недостающие папки
    for name, path in required_dirs.items():
        os.makedirs(path, exist_ok=True)
        print(f"✅ Создана папка: {path}")

    # Пытаемся найти и переместить файлы
    if all_images and all_labels:
        print("\n📦 Распределение файлов по папкам...")

        # Простая стратегия: 80% train, 20% valid
        from sklearn.model_selection import train_test_split

        # Сопоставляем изображения с аннотациями
        matched_pairs = []
        for img in all_images:
            img_name = os.path.splitext(os.path.basename(img))[0]
            label_candidates = [l for l in all_labels if img_name in l]
            if label_candidates:
                matched_pairs.append((img, label_candidates[0]))

        print(f"🔗 Найдено пар (изображение + аннотация): {len(matched_pairs)}")

        if matched_pairs:
            train_pairs, valid_pairs = train_test_split(
                matched_pairs, test_size=0.2, random_state=42
            )

            # Копируем в train
            for img, lbl in train_pairs:
                src_img = os.path.join(base_dir, img)
                src_lbl = os.path.join(base_dir, lbl)
                dst_img = os.path.join(required_dirs['train/images'], os.path.basename(img))
                dst_lbl = os.path.join(required_dirs['train/labels'], os.path.basename(lbl))

                shutil.copy2(src_img, dst_img)
                shutil.copy2(src_lbl, dst_lbl)

            # Копируем в valid
            for img, lbl in valid_pairs:
                src_img = os.path.join(base_dir, img)
                src_lbl = os.path.join(base_dir, lbl)
                dst_img = os.path.join(required_dirs['valid/images'], os.path.basename(img))
                dst_lbl = os.path.join(required_dirs['valid/labels'], os.path.basename(lbl))

                shutil.copy2(src_img, dst_img)
                shutil.copy2(src_lbl, dst_lbl)

            print(f"✅ Train: {len(train_pairs)} пар")
            print(f"✅ Valid: {len(valid_pairs)} пар")
        else:
            print("❌ Не удалось сопоставить изображения с аннотациями!")
    else:
        print("❌ Файлы не найдены для автоматического распределения!")

# ===== 4. ПРОВЕРКА TEST SET =====
print(f"\n" + "=" * 60)
print("🧪 ПРОВЕРКА TEST SET")
print("=" * 60)

test_dirs = {
    'test/images': os.path.join(base_dir, 'test', 'images'),
    'test/labels': os.path.join(base_dir, 'test', 'labels'),
}

has_test = all(os.path.exists(p) for p in test_dirs.values())

if has_test:
    print("✅ Test set существует!")
    for name, path in test_dirs.items():
        count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f"   {name}: {count} файлов")
else:
    print("⚠️  Test set отсутствует!")
    print("💡 Рекомендация: создать test set из части valid данных\n")

    # Создаем test set из valid (опционально)
    create_test = input("Создать test set из 50% valid данных? (y/n): ").lower() == 'y'

    if create_test:
        from sklearn.model_selection import train_test_split

        valid_images_dir = required_dirs['valid/images']
        valid_labels_dir = required_dirs['valid/labels']

        if os.path.exists(valid_images_dir):
            valid_imgs = [f for f in os.listdir(valid_images_dir) if f.endswith(('.jpg', '.png'))]

            if len(valid_imgs) > 1:
                # Создаем папки test
                for path in test_dirs.values():
                    os.makedirs(path, exist_ok=True)

                # Делим valid пополам
                valid_keep, test_imgs = train_test_split(valid_imgs, test_size=0.5, random_state=42)

                # Перемещаем в test
                for img_name in test_imgs:
                    lbl_name = os.path.splitext(img_name)[0] + '.txt'

                    src_img = os.path.join(valid_images_dir, img_name)
                    src_lbl = os.path.join(valid_labels_dir, lbl_name)
                    dst_img = os.path.join(test_dirs['test/images'], img_name)
                    dst_lbl = os.path.join(test_dirs['test/labels'], lbl_name)

                    shutil.move(src_img, dst_img)
                    if os.path.exists(src_lbl):
                        shutil.move(src_lbl, dst_lbl)

                print(f"✅ Test set создан: {len(test_imgs)} файлов")
                print(f"✅ Valid остался: {len(valid_keep)} файлов")

# ===== 5. ФИНАЛЬНАЯ СВОДКА =====
print(f"\n" + "=" * 60)
print("📊 ИТОГОВАЯ СТРУКТУРА")
print("=" * 60)

for name, path in {**required_dirs, **test_dirs}.items():
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
        print(f"✅ {name:20} {count:>4} файлов")
    else:
        print(f"❌ {name:20} не существует")

print("\n🚀 Датасет готов к обучению!")
print("=" * 60)

🔍 ДИАГНОСТИКА ДАТАСЕТА

📄 Содержимое data.yaml:
names:
- detail
- legend
- note
- plan
- section
- site_plan
- stamp
- table
nc: 8
train: train/images
val: valid/images


📂 Проверка структуры папок:
------------------------------------------------------------
✅ train/images         -> /content/datasets/datasets/Project_Test_1-1/train/images
   └─ Файлов: 0
✅ train/labels         -> /content/datasets/datasets/Project_Test_1-1/train/labels
   └─ Файлов: 0
✅ valid/images         -> /content/datasets/datasets/Project_Test_1-1/valid/images
   └─ Файлов: 0
✅ valid/labels         -> /content/datasets/datasets/Project_Test_1-1/valid/labels
   └─ Файлов: 0

🔎 Поиск всех изображений и аннотаций в /content/datasets/datasets/Project_Test_1-1:
------------------------------------------------------------
📸 Найдено изображений: 0
🏷️  Найдено аннотаций: 0

🧪 ПРОВЕРКА TEST SET
⚠️  Test set отсутствует!
💡 Рекомендация: создать test set из части valid данных

Создать test set из 50% valid данных? (y/n): 